# Potter Airlines — SQLite Database

This notebook connects the generated `flights.json` dataset to a SQLite database and provides parameterized CRUD operations.

In [1]:
import sqlite3
import json

DB_NAME = "potter_airlines.db"
JSON_FILE = "flights.json"


## 1. Create the Flights Table

In [2]:
def create_table():
    """Create the flights table if it does not already exist."""

    with sqlite3.connect(DB_NAME) as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS flights (
                flight_id TEXT PRIMARY KEY,
                origin TEXT NOT NULL,
                destination TEXT NOT NULL,
                departure_date TEXT NOT NULL,
                days_until_departure INTEGER NOT NULL,
                base_fare REAL NOT NULL,
                seats_remaining INTEGER NOT NULL,
                capacity INTEGER NOT NULL,
                route_popularity REAL NOT NULL,
                international INTEGER NOT NULL
            )
        """)

## 2. Load Flight Data from JSON

In [3]:
def load_json(filename=JSON_FILE):
    """Load flight data from the JSON file."""

    with open(filename, "r") as file:
        return json.load(file)


## 3. INSERT — One Flight

In [4]:
def insert_flight(flight):
    """Insert one flight into the database."""

    with sqlite3.connect(DB_NAME) as conn:
        conn.execute("""
            INSERT INTO flights
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            flight["flight_id"],
            flight["origin"],
            flight["destination"],
            flight["departure_date"],
            flight["days_until_departure"],
            flight["base_fare"],
            flight["seats_remaining"],
            flight["capacity"],
            flight["route_popularity"],
            flight["international"]
        ))


## 4. Insert All Flights from JSON

In [5]:
def insert_all_flights(flights):
    """Insert all flights from the JSON dataset."""

    with sqlite3.connect(DB_NAME) as conn:
        for flight in flights:
            conn.execute("""
                INSERT OR IGNORE INTO flights
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                flight["flight_id"],
                flight["origin"],
                flight["destination"],
                flight["departure_date"],
                flight["days_until_departure"],
                flight["base_fare"],
                flight["seats_remaining"],
                flight["capacity"],
                flight["route_popularity"],
                flight["international"]
            ))


## 5. SELECT — One Flight

In [6]:
def get_flight(flight_id):
    """Retrieve one flight by flight ID."""

    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.execute("""
            SELECT * FROM flights
            WHERE flight_id = ?
        """, (flight_id,))

        rows = cursor.fetchall()

        if len(rows) == 0:
            print("Flight not found.")

        return rows


## 6. SELECT — All Flights

In [7]:
def get_all_flights():
    """Retrieve all flights from the database."""

    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.execute("""
            SELECT * FROM flights
        """)

        rows = cursor.fetchall()
        return rows

## 7. UPDATE — Seats Remaining

This function updates the number of remaining seats for a selected flight using a parameterized SQL query.

In [8]:
def update_seats(flight_id, new_seats):
    """Update the number of seats remaining for a flight."""

    with sqlite3.connect(DB_NAME) as conn:
        conn.execute("""
            UPDATE flights
            SET seats_remaining = ?
            WHERE flight_id = ?
        """, (new_seats, flight_id))

## 8. UPDATE — Days Until Departure

This function updates the number of days until departure for a selected flight using a parameterized SQL query.

In [9]:
def update_days_until_departure(flight_id, new_days):
    """Update the number of days until departure for a flight."""

    with sqlite3.connect(DB_NAME) as conn:
        conn.execute("""
            UPDATE flights
            SET days_until_departure = ?
            WHERE flight_id = ?
        """, (new_days, flight_id))


## 9. DELETE — One Flight

In [10]:
def delete_flight(flight_id):
    """Delete one flight by flight ID."""

    with sqlite3.connect(DB_NAME) as conn:
        conn.execute("""
            DELETE FROM flights
            WHERE flight_id = ?
        """, (flight_id,))


## 10. Set Up Database and Validate

In [11]:
create_table()

flights = load_json()
insert_all_flights(flights)

print(f"{len(flights)} flights loaded from JSON.")
print(f"{len(get_all_flights())} flights stored in SQLite.")

# Check that all flights were successfully stored
assert len(get_all_flights()) == len(flights)

print("Database setup completed successfully.")


1044 flights loaded from JSON.
1044 flights stored in SQLite.
Database setup completed successfully.


## Summary

The flight data from `flights.json` is stored in a SQLite database called `potter_airlines.db`. The `flights` table follows the same structure as the JSON dataset so that the data can be transferred directly into the database. Parameterized SQL queries are used to insert and retrieve flight records, update remaining seats, and delete flights when needed. Basic database constraints and update validation are included to prevent invalid values such as negative capacity, invalid seat counts, route popularity outside 0–1, or an invalid international indicator. This allows the project to store flight information persistently and access or modify specific records efficiently.
